# 01 — Dataset Audition and Selection

Lightweight, comparable audit of the three candidate datasets supplied for
Assignment 1: `airline_delay.csv`, `hotel_bookings.csv`, `train_occupancy.csv`.
Produces the comparison table used to justify selecting **`hotel_bookings.csv`**
for detailed cleaning and EDA in `02_data_readiness.ipynb`.

No detailed cleaning or full EDA is performed here for any dataset.

In [45]:
import pandas as pd
import numpy as np


In [46]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

file_paths = {
    "airline_delay": "airline_delay.csv",
    "hotel_bookings": "hotel_bookings.csv",
    "train_occupancy": "train_occupancy.csv",
}

dataframes = {name: pd.read_csv(path, low_memory=False) for name, path in file_paths.items()}

for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

airline_delay: 101,000 rows x 29 columns
hotel_bookings: 119,987 rows x 32 columns
train_occupancy: 50,250 rows x 14 columns


In [47]:
df_airline = pd.read_csv("airline_delay.csv", low_memory=False)
df_hotel = pd.read_csv("hotel_bookings.csv", low_memory=False)
df_train = pd.read_csv("train_occupancy.csv", low_memory=False)

# Store in a dictionary to easily loop through your profiling function
dataframes = {
    "airline_delay": df_airline,
    "hotel_bookings": df_hotel,
    "train_occupancy": df_train,
}

## Profile each dataset (size, dtypes, missingness)

In [48]:
def profile_dataset(name, df, top_n_missing=5):
    dtype_counts = df.dtypes.astype(str).value_counts().to_dict()
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    top_missing = missing_pct.head(top_n_missing)
    top_missing_str = ", ".join(f"{col} ({pct:.1f}%)" for col, pct in top_missing.items() if pct > 0)
    n_numeric = df.select_dtypes(include=[np.number]).shape[1]
    n_categorical = df.shape[1] - n_numeric
    return {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "numeric_vars": n_numeric,
        "categorical_vars": n_categorical,
        "dtype_breakdown": dtype_counts,
        "top_missing_columns": top_missing_str if top_missing_str else "None",
        "overall_missing_pct": round(df.isna().mean().mean() * 100, 2),
        "duplicate_rows": int(df.duplicated().sum()),
    }

profiles = [profile_dataset(name, df) for name, df in dataframes.items()]
profile_df = pd.DataFrame(profiles).set_index("dataset")
profile_df

,rows,columns,numeric_vars,categorical_vars,dtype_breakdown,top_missing_columns,overall_missing_pct,duplicate_rows
dataset,,,,,,,,
airline_delay,101000,29,14,15,"{'object': 15, 'float64': 9, 'int64': 5}","CancellationCode (98.4%), NASDelay (80.4%), La...",17.94,1000
hotel_bookings,119987,32,20,12,"{'int64': 16, 'object': 12, 'float64': 4}","company (94.3%), agent (13.8%), country (0.5%)...",3.41,31328
train_occupancy,50250,14,2,12,"{'object': 12, 'int64': 2}","Orig..Station (0.2%), Segment.Direction (0.2%)...",0.07,250


## Quick look at candidate outcome variables
### the candiate outcome for each dataset are :

#### - Airline Delay - ArrDelay
#### - Hotel bookings - is canceled
#### - Train occupancy - Occupancy Status

In [49]:
print("airline_delay -> ArrDelay summary")
print(dataframes["airline_delay"]["ArrDelay"].describe())
print()

print("hotel_bookings -> is_canceled value counts")
print(dataframes["hotel_bookings"]["is_canceled"].value_counts(normalize=True).round(3))
print()

print("train_occupancy -> Occupancy Status value counts")
print(dataframes["train_occupancy"]["Occupancy Status"].value_counts(normalize=True).round(3))

airline_delay -> ArrDelay summary
count    97700.000000
mean         5.711648
std         30.913736
min        -57.000000
25%         -9.000000
50%         -2.000000
75%         10.000000
max        500.000000
Name: ArrDelay, dtype: float64

hotel_bookings -> is_canceled value counts
is_canceled
0    0.63
1    0.37
Name: proportion, dtype: float64

train_occupancy -> Occupancy Status value counts
Occupancy Status
MANY_SEATS_AVAILABLE    0.957
FEW_SEATS_AVAILABLE     0.032
STANDING_ROOM_ONLY      0.008
MANY_SEATSAVAILABLE     0.003
few_seats_available     0.000
VTANDING_ROOM_ONLY      0.000
Name: proportion, dtype: float64


## Quick look at shape of each dataset

In [50]:
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

airline_delay: 101,000 rows x 29 columns
hotel_bookings: 119,987 rows x 32 columns
train_occupancy: 50,250 rows x 14 columns


## this block summarises size, variable types, and top missing columns

In [51]:
def profile_dataset(name, df, top_n_missing=5):
    """Return a dict summarising size, variable types, and top missing columns."""
    dtype_counts = df.dtypes.astype(str).value_counts().to_dict()

    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    top_missing = missing_pct.head(top_n_missing)
    top_missing_str = ", ".join(f"{col} ({pct:.1f}%)" for col, pct in top_missing.items() if pct > 0)

    n_numeric = df.select_dtypes(include=[np.number]).shape[1]
    n_categorical = df.shape[1] - n_numeric

    return {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "numeric_vars": n_numeric,
        "categorical_vars": n_categorical,
        "dtype_breakdown": dtype_counts,
        "top_missing_columns": top_missing_str if top_missing_str else "None",
        "overall_missing_pct": round(df.isna().mean().mean() * 100, 2),
        "duplicate_rows": int(df.duplicated().sum()),
    }

profiles = [profile_dataset(name, df) for name, df in dataframes.items()]
profile_df = pd.DataFrame(profiles).set_index("dataset")
profile_df

,rows,columns,numeric_vars,categorical_vars,dtype_breakdown,top_missing_columns,overall_missing_pct,duplicate_rows
dataset,,,,,,,,
airline_delay,101000,29,14,15,"{'object': 15, 'float64': 9, 'int64': 5}","CancellationCode (98.4%), NASDelay (80.4%), La...",17.94,1000
hotel_bookings,119987,32,20,12,"{'int64': 16, 'object': 12, 'float64': 4}","company (94.3%), agent (13.8%), country (0.5%)...",3.41,31328
train_occupancy,50250,14,2,12,"{'object': 12, 'int64': 2}","Orig..Station (0.2%), Segment.Direction (0.2%)...",0.07,250


## function to check what type of data is present whether is it is nominal, ordinal, interval or ratio

In [52]:
import pandas as pd
import numpy as np

def profile_columns(name, df, measurement_scales):
    results = []

    for column in df.columns:
        dtype = str(df[column].dtype)
        missing_pct = df[column].isna().mean() * 100

        if pd.api.types.is_numeric_dtype(df[column]):
            data_type = "Numeric"
        elif pd.api.types.is_datetime64_any_dtype(df[column]):
            data_type = "Datetime"
        elif pd.api.types.is_bool_dtype(df[column]):
            data_type = "Boolean"
        else:
            data_type = "Categorical/Text"

        scale = measurement_scales.get(column, "Needs classification")

        results.append({
            "Column": column,
            "Data Type": data_type,
            "Pandas Dtype": dtype,
            "Missing %": round(missing_pct, 2),
            "Measurement Scale": scale
        })

    return pd.DataFrame(results)

In [53]:
train_scales = {}

train_profile = profile_columns(
    "train",
    df_train,
    train_scales
)

train_profile

,Column,Data Type,Pandas Dtype,Missing %,Measurement Scale
0,day,Numeric,int64,0.00,Needs classification
1,Actual.Stop.Station,Categorical/Text,object,0.15,Needs classification
2,Actual.Station.Arrv.Time,Categorical/Text,object,0.01,Needs classification
3,Actual.Station.Dprt.Time,Categorical/Text,object,0.02,Needs classification
4,Segment.Direction,Categorical/Text,object,0.16,Needs classification
5,Trip.Name,Categorical/Text,object,0.00,Needs classification
6,Service.Line,Categorical/Text,object,0.12,Needs classification
7,Orig..Station,Categorical/Text,object,0.16,Needs classification
8,Dest..Station,Categorical/Text,object,0.13,Needs classification
9,Leading.Set.Type,Categorical/Text,object,0.15,Needs classification


In [54]:
airline_scales = {

}

airline_profile = profile_columns(
    "airline",
    df_airline,
    airline_scales
)

airline_profile

,Column,Data Type,Pandas Dtype,Missing %,Measurement Scale
0,Year,Numeric,int64,0.00,Needs classification
1,Month,Numeric,int64,0.00,Needs classification
2,DayofMonth,Numeric,int64,0.00,Needs classification
3,DayOfWeek,Numeric,int64,0.00,Needs classification
4,DepTime,Categorical/Text,object,1.12,Needs classification
5,CRSDepTime,Categorical/Text,object,0.00,Needs classification
6,ArrTime,Categorical/Text,object,1.27,Needs classification
7,CRSArrTime,Categorical/Text,object,0.00,Needs classification
8,UniqueCarrier,Categorical/Text,object,0.00,Needs classification
9,FlightNum,Numeric,int64,0.00,Needs classification


In [55]:
hotel_scales = {
    # "Hotel": "Nominal",
    # "Rating": "Ordinal",
    # "Price": "Ratio"
}

hotel_profile = profile_columns(
    "hotel",
    df_hotel,
    hotel_scales
)

hotel_profile

,Column,Data Type,Pandas Dtype,Missing %,Measurement Scale
0,hotel,Categorical/Text,object,0.00,Needs classification
1,is_canceled,Numeric,int64,0.00,Needs classification
2,lead_time,Numeric,int64,0.00,Needs classification
3,arrival_date_year,Numeric,int64,0.00,Needs classification
4,arrival_date_month,Categorical/Text,object,0.00,Needs classification
5,arrival_date_week_number,Numeric,int64,0.00,Needs classification
6,arrival_date_day_of_month,Numeric,int64,0.00,Needs classification
7,stays_in_weekend_nights,Numeric,int64,0.00,Needs classification
8,stays_in_week_nights,Numeric,int64,0.00,Needs classification
9,adults,Numeric,int64,0.00,Needs classification


In [56]:
# Train: Occupancy Status (ordinal categorical target)
print("train_occupancy -> Occupancy Status value counts")
print(dataframes["train_occupancy"]["Occupancy Status"].value_counts(normalize=True).round(3))

train_occupancy -> Occupancy Status value counts
Occupancy Status
MANY_SEATS_AVAILABLE    0.957
FEW_SEATS_AVAILABLE     0.032
STANDING_ROOM_ONLY      0.008
MANY_SEATSAVAILABLE     0.003
few_seats_available     0.000
VTANDING_ROOM_ONLY      0.000
Name: proportion, dtype: float64


In [57]:
# Airline: ArrDelay (regression/classification target)
print("airline_delay -> ArrDelay summary")
print(dataframes["airline_delay"]["ArrDelay"].describe())
print()

airline_delay -> ArrDelay summary
count    97700.000000
mean         5.711648
std         30.913736
min        -57.000000
25%         -9.000000
50%         -2.000000
75%         10.000000
max        500.000000
Name: ArrDelay, dtype: float64



In [58]:
# Hotel: is_canceled (binary classification target)
print("hotel_bookings -> is_canceled value counts")
print(dataframes["hotel_bookings"]["is_canceled"].value_counts(normalize=True).round(3))
print()


hotel_bookings -> is_canceled value counts
is_canceled
0    0.63
1    0.37
Name: proportion, dtype: float64



In [59]:
comparison = pd.DataFrame([
    {
        "Dataset": "airline_delay.csv",
        "Size": "101,000 rows x 29 vars",
        "Variable types": "5 int, 9 float, 15 categorical/string",
        "Outcome / opportunity": "Predict ArrDelay (regression) or delay probability (classification)",
        "Major data-quality risks": "~80% missing on 4 delay-cause columns; 98% missing CancellationCode",
        "Stakeholder decision": "Ops manager targeting investment to improve on-time performance",
        "Ethical/feasibility concern": "Delay attribution may unfairly shift blame; data from 2008 may be outdated",
    },
    {
        "Dataset": "hotel_bookings.csv",
        "Size": "119,987 rows x 32 vars",
        "Variable types": "16 int, 4 float, 12 categorical/string",
        "Outcome / opportunity": "Predict is_canceled (classification) or ADR/revenue",
        "Major data-quality risks": "94% missing company, 14% missing agent; known outliers in adr/lead_time",
        "Stakeholder decision": "Revenue manager setting deposit/overbooking policy to cut cancellations",
        "Ethical/feasibility concern": "agent/company/country fields risk re-identifying agents or guests",
    },
    {
        "Dataset": "train_occupancy.csv",
        "Size": "50,250 rows x 14 vars",
        "Variable types": "2 int, 12 categorical/string (mostly text/ordinal)",
        "Outcome / opportunity": "Classify Occupancy Status by time band, line, station",
        "Major data-quality risks": "Timestamps stored as strings; ordinal (not count) target; few numeric features",
        "Stakeholder decision": "Transit planner allocating peak-time carriage capacity",
        "Ethical/feasibility concern": "Station-level data may reveal travel patterns of specific communities",
    },
]).set_index("Dataset")

comparison

,Size,Variable types,Outcome / opportunity,Major data-quality risks,Stakeholder decision,Ethical/feasibility concern
Dataset,,,,,,
airline_delay.csv,"101,000 rows x 29 vars","5 int, 9 float, 15 categorical/string",Predict ArrDelay (regression) or delay probabi...,~80% missing on 4 delay-cause columns; 98% mis...,Ops manager targeting investment to improve on...,Delay attribution may unfairly shift blame; da...
hotel_bookings.csv,"119,987 rows x 32 vars","16 int, 4 float, 12 categorical/string",Predict is_canceled (classification) or ADR/re...,"94% missing company, 14% missing agent; known ...",Revenue manager setting deposit/overbooking po...,agent/company/country fields risk re-identifyi...
train_occupancy.csv,"50,250 rows x 14 vars","2 int, 12 categorical/string (mostly text/ordi...","Classify Occupancy Status by time band, line, ...",Timestamps stored as strings; ordinal (not cou...,Transit planner allocating peak-time carriage ...,Station-level data may reveal travel patterns ...


In [60]:
comparison.to_csv("eda_comparison_table.csv")
print("Saved to eda_comparison_table.csv")

Saved to eda_comparison_table.csv


## 6. Dataset selection

**Selected dataset: `hotel_bookings.csv`**

Justification (evidence from the code above):


1. **Lots of useful variables** — 32 columns covering numbers, yes/no fields, and
   categories, which is enough to build a clear target: whether a booking gets
   cancelled (`is_canceled`).
2. **Missing data is manageable** — the important columns (lead_time, adr,
   market_segment, deposit_type) are almost fully filled in. Only `company` and
   `agent` have a lot of gaps, and those can just be dropped or flagged.
3. **Useful for a real decision** — knowing which bookings are likely to cancel
   helps a hotel decide things like deposit rules or how much to overbook.
4. **Fewer privacy issues** — the columns that could identify people (agent,
   company) aren't needed for the analysis, so they can just be dropped.
